In [2]:

eval_questions = [
    # Answerable questions: expected source is the PDF filename

    {
        "id": 1,
        "question": "How does LoRA reduce the number of trainable parameters?",
        "expected_source": "Lora.pdf",
        "answerable": True,
    },
    {
        "id": 2,
        "question": "How does self-attention compute relationships between tokens?",
        "expected_source": "attention-is-all-you-need-Paper.pdf",
        "answerable": True,
    },
    {
        "id": 3,
        "question": "What is the purpose of positional encoding in the Transformer?",
        "expected_source": "attention-is-all-you-need-Paper.pdf",
        "answerable": True,
    },
    {
        "id": 4,
        "question": "How does chain-of-thought prompting help language models solve problems?",
        "expected_source": "chain-of-thought.pdf",
        "answerable": True,
    },
    {
        "id": 5,
        "question": "What problem do residual connections address in deep neural networks?",
        "expected_source": "deep residual learning for image recognition.pdf",
        "answerable": True,
    },
    {
        "id": 6,
        "question": "What are the roles of the generator and discriminator in GANs?",
        "expected_source": "GANs.pdf",
        "answerable": True,
    },

    # More specific questions: useful for testing retrieval difficulty

    {
        "id": 7,
        "question": "How does LoRA represent a weight update using low-rank matrices?",
        "expected_source": "Lora.pdf",
        "answerable": True,
    },
    {
        "id": 8,
        "question": "Why does a Transformer use multiple attention heads?",
        "expected_source": "attention-is-all-you-need-Paper.pdf",
        "answerable": True,
    },
    {
        "id": 9,
        "question": "How does the residual learning formulation help train very deep networks?",
        "expected_source": "deep residual learning for image recognition.pdf",
        "answerable": True,
    },
    {
        "id": 10,
        "question": "How is adversarial training used in a generative adversarial network?",
        "expected_source": "GANs.pdf",
        "answerable": True,
    },

    # Unanswerable questions: information expected to be absent from the PDFs

    {
        "id": 11,
        "question": "What exact GPU model and batch size were used in my latest training run?",
        "expected_source": None,
        "answerable": False,
    },
    {
        "id": 12,
        "question": "What is the current market price of NVIDIA stock?",
        "expected_source": None,
        "answerable": False,
    },
    {
        "id": 13,
        "question": "What final examination marks did the authors receive in university?",
        "expected_source": None,
        "answerable": False,
    },
]

print(f"Total evaluation questions: {len(eval_questions)}")
print(
    "Answerable questions:",
    sum(q["answerable"] for q in eval_questions),
)
print(
    "Unanswerable questions:",
    sum(not q["answerable"] for q in eval_questions),
)

Total evaluation questions: 13
Answerable questions: 10
Unanswerable questions: 3


In [3]:

import sys
from pathlib import Path

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loader import load_pdf
from src.chunker import create_page_chunks
from src.retriever import build_bm25

# Load PDFs and preserve document metadata
data_dir = PROJECT_ROOT / "data"
pdf_files = sorted(data_dir.glob("*.pdf"))

all_pages = []

for pdf_path in pdf_files:
    pages = load_pdf(pdf_path)

    for page in pages:
        page["source"] = pdf_path.name

    all_pages.extend(pages)

print("PDF files loaded:", len(pdf_files))
print("Pages loaded:", len(all_pages))

# Create chunks
chunks = create_page_chunks(
    all_pages,
    chunk_size=1500,
    overlap_paragraphs=1,
)

print("Total chunks:", len(chunks))

# Load embedding model
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# Build FAISS index using normalized embeddings
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype("float32")

faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# Build BM25 keyword index
bm25_module = build_bm25(chunks)

# Load cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("\nRetrieval components initialized successfully.")
print("FAISS vectors:", index.ntotal)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/KGMNOB+TT93o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(64, 0, 2204658414928), '/LastChar': 121, '/Widths': [333, 0, 0, 507, 0, 507, 0, 0, 507, 507, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 440, 0, 0, 0, 440, 0, 0, 0, 0, 0, 0, 280, 0, 0, 0, 0, 0, 333, 0, 0, 0, 0, 0, 0, 480]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/MKDMEG+TT55o00', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 45, '/FontDescriptor': IndirectObject(312, 0, 2204658414928), '/LastChar': 121, '/Widths': [340, 0, 0, 0, 500, 0, 500, 500, 0, 0, 0, 500, 0, 0, 0, 0, 

PDF files loaded: 5
Pages loaded: 72
Total chunks: 76


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Retrieval components initialized successfully.
FAISS vectors: 76


In [4]:

assert len(chunks) > 0, "No chunks were created."
assert index.ntotal == len(chunks), "FAISS index size mismatch."
assert len(pdf_files) > 0, "No PDF files found."

print("Setup verification passed.")
print("Documents:", len(pdf_files))
print("Chunks:", len(chunks))
print("FAISS vectors:", index.ntotal)

Setup verification passed.
Documents: 5
Chunks: 76
FAISS vectors: 76


In [5]:

import pandas as pd

from src.retriever import (
    dense_retrieve,
    hybrid_retrieve,
    rerank,
)

# Evaluate only questions with known expected source documents
answerable_questions = [
    q for q in eval_questions if q["answerable"]
]

K_VALUES = [1, 3, 5]


def calculate_retrieval_metrics(results, expected_source):
    """Calculate Recall@K and reciprocal rank at the document level."""
    ranked_sources = [result["source"] for result in results]

    relevant_ranks = [
        rank
        for rank, source in enumerate(ranked_sources, start=1)
        if source == expected_source
    ]

    metrics = {}

    for k in K_VALUES:
        metrics[f"Recall@{k}"] = int(
            expected_source in ranked_sources[:k]
        )

    metrics["Reciprocal Rank"] = (
        1.0 / relevant_ranks[0] if relevant_ranks else 0.0
    )

    return metrics


def evaluate_retrieval_strategies():
    """Compare dense, hybrid, and hybrid + reranking."""
    records = []

    for question_data in answerable_questions:
        query = question_data["question"]
        expected_source = question_data["expected_source"]

        # Strategy 1: Dense retrieval
        dense_results = dense_retrieve(
            query=query,
            model=model,
            index=index,
            chunks=chunks,
            k=5,
        )

        # Strategy 2: Hybrid retrieval (dense + BM25 + RRF)
        hybrid_results = hybrid_retrieve(
            query=query,
            model=model,
            index=index,
            bm25=bm25_module,
            chunks=chunks,
            dense_k=10,
            bm25_k=10,
            final_k=5,
        )

        # Strategy 3: Hybrid retrieval + cross-encoder reranking
        hybrid_candidates = hybrid_retrieve(
            query=query,
            model=model,
            index=index,
            bm25=bm25_module,
            chunks=chunks,
            dense_k=10,
            bm25_k=10,
            final_k=10,
        )

        reranked_results = rerank(
            query=query,
            candidates=hybrid_candidates,
            reranker=reranker,
            top_k=5,
        )

        strategies = {
            "Dense": dense_results,
            "Hybrid": hybrid_results,
            "Hybrid + Reranking": reranked_results,
        }

        for strategy_name, results in strategies.items():
            metrics = calculate_retrieval_metrics(
                results, expected_source
            )

            records.append({
                "Question ID": question_data["id"],
                "Strategy": strategy_name,
                "Expected Source": expected_source,
                **metrics,
            })

    return pd.DataFrame(records)


evaluation_df = evaluate_retrieval_strategies()

print("Evaluation completed.")
print("Questions evaluated:", len(answerable_questions))
display(evaluation_df)

Evaluation completed.
Questions evaluated: 10


,Question ID,Strategy,Expected Source,Recall@1,Recall@3,Recall@5,Reciprocal Rank
0,1,Dense,Lora.pdf,1,1,1,1.0
1,1,Hybrid,Lora.pdf,1,1,1,1.0
2,1,Hybrid + Reranking,Lora.pdf,1,1,1,1.0
3,2,Dense,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
4,2,Hybrid,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
5,2,Hybrid + Reranking,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
6,3,Dense,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
7,3,Hybrid,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
8,3,Hybrid + Reranking,attention-is-all-you-need-Paper.pdf,1,1,1,1.0
9,4,Dense,chain-of-thought.pdf,1,1,1,1.0


In [6]:

from src.pipeline import answer_question

unanswerable_questions = [
    q for q in eval_questions if not q["answerable"]
]

abstention_results = []

for item in unanswerable_questions:
    result = answer_question(
        query=item["question"],
        model=model,
        index=index,
        bm25=bm25_module,
        chunks=chunks,
        reranker=reranker,
    )

    abstention_results.append({
        "Question ID": item["id"],
        "Question": item["question"],
        "Answer": result["answer"],
        "Retrieved Sources": [
            source["source"] for source in result["sources"]
        ],
    })

abstention_df = pd.DataFrame(abstention_results)

display(abstention_df)

,Question ID,Question,Answer,Retrieved Sources
0,11,What exact GPU model and batch size were used ...,"I’m sorry, but the provided documents do not c...","[Lora.pdf, Lora.pdf, attention-is-all-you-need..."
1,12,What is the current market price of NVIDIA stock?,"I’m sorry, but the provided documents do not c...","[Lora.pdf, attention-is-all-you-need-Paper.pdf..."
2,13,What final examination marks did the authors r...,The provided documents contain only technical ...,"[Lora.pdf, Lora.pdf, GANs.pdf]"


In [7]:

# Fill these values after reviewing each answer.
# True = the model appropriately acknowledged insufficient evidence.
# False = the model attempted an unsupported answer.

manual_abstention_labels = [
    # Example: True, False, True
]

if len(manual_abstention_labels) != len(unanswerable_questions):
    print(
        f"Please label all {len(unanswerable_questions)} answers first."
    )
else:
    abstention_rate = sum(manual_abstention_labels) / len(
        manual_abstention_labels
    )

    print(f"Correct abstentions: {sum(manual_abstention_labels)}")
    print(f"Total unanswerable questions: {len(manual_abstention_labels)}")
    print(f"Abstention rate: {abstention_rate:.2%}")

Please label all 3 answers first.


In [8]:

# Based on the three answers you inspected.
# Change a label to False if its full answer makes unsupported claims.

manual_abstention_labels = [True, True, True]

abstention_rate = (
    sum(manual_abstention_labels)
    / len(manual_abstention_labels)
)

print("Unanswerable questions:", len(manual_abstention_labels))
print("Correct abstentions:", sum(manual_abstention_labels))
print(f"Abstention rate: {abstention_rate:.1%}")

Unanswerable questions: 3
Correct abstentions: 3
Abstention rate: 100.0%


In [9]:

from src.retriever import retrieve_relevant_chunks
from src.generator import generate_answer
from src.evaluator import evaluate_citations

query = "How does self-attention work?"

# Retrieve evidence
retrieved_chunks = retrieve_relevant_chunks(
    query=query,
    model=model,
    index=index,
    bm25=bm25_module,
    chunks=chunks,
    reranker=reranker,
    top_k=3
)

# Generate answer using exactly this evidence
answer = generate_answer(
    query=query,
    results=retrieved_chunks
)

# Evaluate citations
citation_results = evaluate_citations(
    answer=answer,
    retrieved_chunks=retrieved_chunks
)

print("QUESTION:")
print(query)

print("\nGENERATED ANSWER:")
print(answer)

print("\nCITATION EVALUATION:")
for metric, value in citation_results.items():
    print(f"{metric}: {value}")

QUESTION:
How does self-attention work?

GENERATED ANSWER:
**Self‑attention in the Transformer**

| Step | What happens | Where it is used |
|------|--------------|------------------|
| 1. Linear projections | For each head *i* the input is linearly projected to three vectors: <br>  *Q* = *X* W<sub>Qᵢ</sub>, *K* = *X* W<sub>Kᵢ</sub>, *V* = *X* W<sub>Vᵢ</sub> ( W matrices are learned, with dimensions d<sub>model</sub> × d<sub>k</sub> or d<sub>v</sub>)【attention-is-all-you-need-Paper.pdf, p. 5】. |
| 2. Scaled dot‑product attention | For each head the attention weights are computed as <br>  softmax( (Q Kᵀ) / √d<sub>k</sub> ) and applied to *V*: <br>  head<sub>i</sub> = Attention(QW<sub>Qᵢ</sub>, KW<sub>Kᵢ</sub>, VW<sub>Vᵢ</sub>)【attention-is-all-you-need-Paper.pdf, p. 5】. |
| 3. Multi‑head concatenation | The outputs of the *h* heads are concatenated and linearly transformed: <br>  MultiHead(Q,K,V) = Concat(head₁,…,head<sub>h</sub>) W<sub>O</sub>【attention-is-all-you-need-Paper.pdf, p. 5】

In [10]:
from src.evaluator import evaluate_grounding

grounding_results = evaluate_grounding(
    answer=answer,
    retrieved_chunks=retrieved_chunks,
)

print("GROUNDING EVALUATION:")

for metric, value in grounding_results.items():
    print(f"{metric}: {value}")

GROUNDING EVALUATION:
grounding_score: 0.76
supported_terms: 133
total_terms: 175


In [13]:
from src.evaluator import evaluate_citations, evaluate_grounding
from src.retriever import retrieve_relevant_chunks
from src.generator import generate_answer

answer_evaluation = []

for item in answerable_questions:

    query = item["question"]

    # Retrieve evidence
    retrieved_chunks = retrieve_relevant_chunks(
        query=query,
        model=model,
        index=index,
        bm25=bm25_module,
        chunks=chunks,
        reranker=reranker,
        top_k=3,
    )

    # Generate answer
    generated_answer = generate_answer(
        query=query,
        results=retrieved_chunks,
    )

    # Citation evaluation
    citation_metrics = evaluate_citations(
        answer=generated_answer,
        retrieved_chunks=retrieved_chunks,
    )

    # Grounding evaluation
    grounding_metrics = evaluate_grounding(
        answer=generated_answer,
        retrieved_chunks=retrieved_chunks,
    )

    # Check whether expected document was retrieved
    retrieved_sources = [
        chunk["source"]
        for chunk in retrieved_chunks
    ]

    expected_source_retrieved = (
        item["expected_source"] in retrieved_sources
    )

    answer_evaluation.append({
        "Question ID": item["id"],
        "Expected Source": item["expected_source"],
        "Expected Source Retrieved": expected_source_retrieved,
        "Total Citations": citation_metrics["total_citations"],
        "Valid Citations": citation_metrics["valid_citations"],
        "Citation Precision": citation_metrics["citation_precision"],
        "Grounding Score": grounding_metrics["grounding_score"],
    })

answer_evaluation_df = pd.DataFrame(answer_evaluation)

display(
    answer_evaluation_df.round(3)
)

,Question ID,Expected Source,Expected Source Retrieved,Total Citations,Valid Citations,Citation Precision,Grounding Score
0,1,Lora.pdf,True,0,0,0.0,0.832
1,2,attention-is-all-you-need-Paper.pdf,True,0,0,0.0,0.713
2,3,attention-is-all-you-need-Paper.pdf,True,1,0,0.0,0.692
3,4,chain-of-thought.pdf,True,0,0,0.0,0.704
4,5,deep residual learning for image recognition.pdf,True,0,0,0.0,0.778
5,6,GANs.pdf,True,0,0,0.0,0.643
6,7,Lora.pdf,True,0,0,0.0,0.700
7,8,attention-is-all-you-need-Paper.pdf,True,0,0,0.0,0.662
8,9,deep residual learning for image recognition.pdf,True,3,0,0.0,0.708
9,10,GANs.pdf,True,4,4,1.0,0.693


In [14]:
overall_metrics = {
    "Document Retrieval Accuracy":
        answer_evaluation_df["Expected Source Retrieved"].mean(),

    "Average Citation Precision":
        answer_evaluation_df["Citation Precision"].mean(),

    "Average Grounding Score":
        answer_evaluation_df["Grounding Score"].mean(),

    "Questions With Citations":
        (
            answer_evaluation_df["Total Citations"] > 0
        ).mean(),
}

overall_df = pd.DataFrame(
    {
        "Metric": overall_metrics.keys(),
        "Score": overall_metrics.values(),
    }
)

display(overall_df.round(3))

,Metric,Score
0,Document Retrieval Accuracy,1.000
1,Average Citation Precision,0.100
2,Average Grounding Score,0.713
3,Questions With Citations,0.300


In [15]:
for item in answerable_questions:
    query = item["question"]

    retrieved = retrieve_relevant_chunks(
        query=query,
        model=model,
        index=index,
        bm25=bm25_module,
        chunks=chunks,
        reranker=reranker,
        top_k=3,
    )

    generated = generate_answer(
        query=query,
        results=retrieved,
    )

    print("=" * 80)
    print(f"Q{item['id']}: {query}")
    print("\n", generated)

Q1: How does LoRA reduce the number of trainable parameters?

 **Mechanism**

- LoRA **freezes** all original pretrained weights of the Transformer.  
- It **injects** a pair of low‑rank matrices \(A\) and \(B\) (with rank \(r\ll d_{\text{model}}\)) into each target weight matrix, so that only the parameters of \(A\) and \(B\) are updated during downstream training.  
- Because the number of elements in \(A\) and \(B\) is proportional to \(r\cdot d_{\text{model}}\) rather than \(d_{\text{model}}^{2}\), the set of trainable parameters is reduced by several orders of magnitude (e.g., ≈10,000 × fewer for GPT‑3 175B) and the optimizer state memory is saved for the frozen parameters.  

**Evidence**

- “LoRA … freezes the pretrained model weights and injects trainable rank‑decomposition matrices … greatly reducing the number of trainable parameters” [**Lora.pdf, p. 1**]  
- “We only train \(A\) and \(B\). … This allows us to train some dense layers … by optimizing rank decomposition matrice

APIConnectionError: Connection error.